# Day 8 — RAG from Scratch
---
> **Phase:** 2 — RAG Systems  
> **Status:** ✅ Complete

## 🧠 Key Concepts

### 1. Why RAG Exists — The Core Failure Modes

**Problem 1 — Attention Dilution (Lost in the Middle)**
- Stuffing a full document into context overwhelms the model
- Transformers struggle with long contexts — attention spreads thin
- Information in the middle of a long context gets ignored
- Solution: Only feed the model the relevant chunks, not the whole document

**Problem 2 — Token Limits**
- Context windows are finite (even 128k token models have limits)
- A 500-page legal document cannot fit in any context window
- Solution: Retrieve only what's needed per query

**Problem 3 — Hallucination Without Grounding**
- LLM asked about a document it hasn't seen → invents plausible-sounding answer
- Looks correct, is wrong, impossible to detect without grounding
- Solution: RAG + explicit grounding instruction

### 2. The Cross-Reference Failure Mode

**Scenario:** Legal contract. Section 3.2 says:
> *"The liability cap described in Section 7.4 applies to all parties."*

Section 7.4 is 40 pages away.

**What happens with fixed chunking:**
```
Retriever fetches chunk containing Section 3.2
LLM sees: "liability cap applies to all parties"
LLM does NOT see: the actual cap amount in Section 7.4
LLM answers "who" correctly, invents "what" confidently
User gets a wrong answer that looks completely correct
```

**Why hallucination is more dangerous than refusal:**
- Refusal: "I don't have enough information" → user knows to look elsewhere
- Hallucination: "The cap is $500,000" with full confidence → user trusts it, acts on it

**Two production patterns to handle cross-references:**

| Pattern | What it is | When available |
|---------|-----------|----------------|
| Agentic RAG | LLM identifies cross-reference, triggers another retrieval | Phase 3 |
| Document Graph Indexing | Preserve relationship structure as metadata before chunking | LlamaIndex Knowledge Graph |

> **Key insight:** Chunking is not a neutral operation. Every chunking decision destroys some information. Your job is to decide which information you can afford to lose.

### 3. The Complete RAG Pipeline

```
INDEX TIME (happens once)
──────────────────────────────────────────
Step 1 → Load document from disk
Step 2 → Chunk document (size + overlap)
Step 3 → Embed all chunks → store in FAISS

QUERY TIME (happens every request)
──────────────────────────────────────────
Step 4 → User asks question → embed query
Step 5 → Retrieve top-K similar chunks from FAISS
Step 6 → Assemble grounded prompt
          system: grounding instruction
          user:   context + question
Step 7 → Call LLM → return grounded answer
```

**The critical distinction:** Stages 1-3 are paid once. Stages 4-7 are paid per query. Never re-embed on every request.

### 4. Chunking — chunk_size and chunk_overlap

**chunk_size:** How many words per chunk. Controls how much context each chunk carries.

**chunk_overlap:** How many words from the previous chunk bleed into the next. Prevents orphaned sentence fragments at boundaries.

**The boundary problem without overlap:**
```
Sentence: "Therefore, given everything discussed above, the defendant is not liable."

Chunk 4 ends: "Therefore, given everything discussed above,"
Chunk 5 starts: "the defendant is not liable."

→ Chunk 5 has conclusion with no reasoning
→ LLM gets answer without justification
→ Answer is incomplete or wrong
```

**With overlap:**
```
Chunk 5 starts: "...given everything discussed above, the defendant is not liable."
→ Reasoning is preserved at chunk boundary
→ Answer is complete
```

**The sliding window mechanic:**
```
chunk_size = 200, chunk_overlap = 50
step = chunk_size - chunk_overlap = 150

Chunk 0: words[0:200]
Chunk 1: words[150:350]   ← 50 word overlap with chunk 0
Chunk 2: words[300:500]   ← 50 word overlap with chunk 1
```

### 5. Prompt Assembly — The Grounding Pattern

**Three components every RAG prompt must have:**
```
System message → role + grounding instruction
Context        → retrieved chunks joined into one string
Question       → the user's actual query
```

**The grounding instruction:**
> *"Answer only using the provided context. If the answer is not present in the context, say 'I don't have enough information to answer this.' Do not use outside knowledge."*

**Without grounding:** LLM treats retrieved context as a hint, fills gaps from training data → hallucination

**With grounding:** LLM treats retrieved context as the only allowed source → refusal when answer is absent

**Why refusal is valuable in production:**
- Medical tool: "I don't have enough information" → user consults a doctor
- Legal tool: "I don't have enough information" → user reads the actual contract
- Both are infinitely safer than a confident wrong answer

**Chunk sync rule:**
```
chunks list:   [chunk0,  chunk1,  chunk2 ...]
FAISS index:   [vec0,    vec1,    vec2   ...]
                 ↑         ↑        ↑
             same integer position = same document

FAISS returns index 2 → chunks[2] gives the text back
Never shuffle one without shuffling the other
```

## 💻 Code

In [ ]:
# Day 8 — Full RAG Pipeline from Scratch
from dotenv import load_dotenv
from groq import Groq
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import os

load_dotenv()
model = SentenceTransformer("all-MiniLM-L6-v2")
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
# Stage 1 — Load and Chunk
def load_and_chunk(filepath, chunk_size=200, chunk_overlap=50):
    with open(filepath, 'r', encoding='utf-8') as file:
        words = file.read().split()
    
    chunks = []
    step = chunk_size - chunk_overlap
    
    for position in range(0, len(words), step):
        window = words[position : position + chunk_size]
        chunk = " ".join(window)
        chunks.append(chunk)
    
    return chunks

# Why position + chunk_size:
# Start at position, take exactly chunk_size words
# words[0:200], words[150:350], words[300:500]...
# step = chunk_size - chunk_overlap ensures overlap between chunks

In [ ]:
# Stage 2 — Embed and Index
def build_index(chunks):
    doc_vectors = model.encode(chunks)
    doc_vectors = doc_vectors.astype(np.float32)
    
    dimension = doc_vectors.shape[1]  # 384 for all-MiniLM-L6-v2
    index = faiss.IndexFlatL2(dimension)
    # IndexFlatL2 takes dimension, not vectors
    # dimension declared upfront so FAISS can enforce consistency
    
    index.add(doc_vectors)
    print(f"Index built. Vectors stored: {index.ntotal}")
    
    return index, chunks
    # Return BOTH — index for search, chunks for text lookup
    # They stay in sync: FAISS index 2 → chunks[2]

In [ ]:
# Stage 3 — Retrieve
def retrieve(query, index, chunks, top_k=3):
    query_vector = model.encode([query])          # [query] not query — shape rule
    query_vector = query_vector.astype(np.float32)
    
    distances, indices = index.search(query_vector, top_k)
    # distances[0] → array of L2 distances
    # indices[0]   → array of positions in chunks list
    
    results = [(dist, idx) for dist, idx in zip(distances[0], indices[0])]
    
    matching_chunks = []
    for dist, idx in results:
        matching_chunks.append(chunks[idx])
    
    return matching_chunks
    # Returns list of chunk strings, not indices
    # Caller gets text, not positions

In [ ]:
# Stage 4 — Generate (Grounded)
def generate(query, retrieved_chunks):
    context = " ".join(retrieved_chunks)
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "Answer only using the provided context. If the answer is not present in the context, say 'I don't have enough information to answer this.' Do not use outside knowledge."
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {query}"
            }
        ],
        temperature=0,
        max_tokens=500
    )
    
    return response.choices[0].message.content
    # temperature=0 for factual, grounded answers
    # \n\n between context and question helps LLM parse structure

In [ ]:
# Full Pipeline — Wire Everything Together
chunks = load_and_chunk('sample_doc.txt')
print(f"Total chunks: {len(chunks)}")
print(f"Chunk 0 preview: {chunks[0][:200]}...")
print()

index, chunks = build_index(chunks)
print()

# Test 1 — Answer exists in document
query1 = "What are the symptoms of PCOD?"
retrieved1 = retrieve(query1, index, chunks)
answer1 = generate(query1, retrieved1)
print(f"Query: {query1}")
print(f"Answer: {answer1}")
print()

# Test 2 — Answer does NOT exist in document (grounding test)
query2 = "What is the treatment cost of PCOD in India?"
retrieved2 = retrieve(query2, index, chunks)
answer2 = generate(query2, retrieved2)
print(f"Query: {query2}")
print(f"Answer: {answer2}")

## 🔬 Observations

| Test | Query | Result | Lesson |
|------|-------|--------|--------|
| Retrieval | "symptoms of PCOD?" | Correct chunk returned | Semantic search working |
| Generation | "symptoms of PCOD?" | Listed 12 symptoms accurately | Grounded answer from context |
| Grounding | "treatment cost in India?" | "I don't have enough information" | Grounding instruction prevents hallucination |
| Chunking | chunk_size=200, overlap=50 | 6 chunks from document | Overlap preserves boundary context |
| Sync | FAISS index + chunks list | indices[i] → chunks[i] | Integer position is the link |

## ✅ Day 8 Summary

```
✓ RAG failure modes — attention dilution, token limits, hallucination
✓ Cross-reference failure — why fixed chunking breaks legal/technical docs
✓ Hallucination vs refusal — why refusal is safer in production
✓ Agentic RAG pattern (derived independently) — Phase 3 territory
✓ Document graph indexing pattern (derived independently) — LlamaIndex KG
✓ Chunking mechanics — chunk_size, chunk_overlap, sliding window
✓ Why chunk_overlap exists — prevents orphaned sentence fragments
✓ Sliding window math — step = chunk_size - chunk_overlap
✓ Slice arithmetic — words[position : position + chunk_size]
✓ Chunk-FAISS sync rule — integer position links both structures
✓ FAISS shape rule — model.encode([query]) not model.encode(query)
✓ IndexFlatL2 takes dimension not vectors — why dimension declared upfront
✓ Grounding instruction — anchors LLM to retrieved context only
✓ Prompt assembly — system + context + question structure
✓ Full RAG pipeline built from scratch — load, chunk, embed, retrieve, generate
✓ Grounding test passed — refused to answer when info not in document
```

## ⚠️ Production Limitations of Today's Pipeline

```
1. No persistence — index rebuilt every run (Day 5 pattern solves this)
2. Word-level chunking — splits on spaces, not sentences or paragraphs
   Better: sentence-aware chunking using nltk or spacy
3. No distance threshold — retrieves top_k regardless of relevance score
   Better: filter by L2 distance threshold like Day 5
4. No metadata — chunks carry no source info (page number, section name)
   Better: store metadata alongside chunks for citation
5. Single document only — real systems handle hundreds of documents
   Better: batch loading, multi-document indexing
```

These are exactly the gaps Days 9-13 will close.

## 🔜 Bridge to Day 9

Today you chunked by word count — a blunt instrument. 

Think about this before Day 9:

Your document has natural structure — headings, paragraphs, numbered sections. Word-level chunking ignores all of it. A chunk boundary can fall right in the middle of a numbered symptom, splitting `"1. Irregular Periods"` from its explanation.

**What would a smarter chunking strategy look like?** What signals already exist in the document text that you could use to find better split points — ones that respect the document's own structure rather than imposing an arbitrary word count?